# __Chapter 6. Roots: Open Methods__

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### [Algorithm] 단순 고정점 반복법

In [ ]:
import math

def fixed_point(g, x0, tol_percent=0.01, max_iter=10_000, verbose=True,
                x_true=None):
    """
    고정점 반복법 (Fixed-Point Iteration)

    Parameters
    ----------
    g : callable
        고정점식 g(x)
    x0 : float
        초기 추정값
    tol_percent : float
        근사 백분율 상대오차 허용치 (%)
    max_iter : int
        최대 반복 횟수
    verbose : bool
        True이면 반복 과정을 출력
    x_true : float, optional
        참값. 주어지면 각 반복마다 참백분율상대오차(et)와, 이전 반복 대비
        현재 반복의 참오차 비율(et_ratio = et_k / et_(k-1), 백분율)도 함께
        출력한다. 주어지지 않으면(기본값 None) 이 두 값은 계산도 출력도
        하지 않는다.

    Returns
    -------
    root : float
        근사 해
    iters : int
        반복 횟수
    ea : float
        마지막 근사 백분율 상대오차 (%)

    Notes
    -----
    - iteration k의 행은 x_k, g(x_k)를 보여주고, ea_k = |x_k - x_(k-1)| /
      x_k * 100 로 계산한다. 즉 어떤 g(x)를 쓰든 "그 행의 x"와 "그 행의
      오차"는 항상 같은 반복 번호(k)에 대응한다.
    - g(x)가 고정점 수렴 조건(|g'(x*)| < 1)을 만족하지 않으면 반복이
      발산할 수 있다. x가 g(x)의 정의역을 벗어나면(예: log(음수)) 원래의
      알기 힘든 예외 대신 "발산으로 추정됨"을 알리는 RuntimeError를
      발생시킨다. max_iter 안에 수렴하지 못했고 마지막 오차가 첫 오차보다
      커졌다면(오차가 줄지 않고 커지는 추세) 발산 경고 메시지도 출력한다.
    """

    eps = 1e-15

    def true_error(xv):
        denom = x_true if abs(x_true) > eps else eps
        return abs((x_true - xv) / denom) * 100.0

    def safe_g(xv):
        try:
            return g(xv)
        except Exception as exc:
            raise RuntimeError(
                f"g(x) 계산 중 오류가 발생했습니다 (x={xv!r}). "
                "이 g(x)로는 반복법이 발산하고 있을 가능성이 높습니다 "
                "(수렴 조건 |g'(x*)| < 1 을 만족하는지 확인하세요). "
                f"원래 오류: {type(exc).__name__}: {exc}"
            ) from exc

    # Header
    if verbose:
        header = (
            f"{'iter':>4} | "
            f"{'x':>10} | "
            f"{'g(x)':>10} | "
            f"{'ea(%)':>8}"
        )
        if x_true is not None:
            header += f" | {'et(%)':>8} | {'et_ratio(%)':>12}"
        print(header)
        print("-" * len(header))

    x_prev = None
    x = float(x0)
    ea = None
    ea_first = None
    et = None
    et_prev = None

    # iteration k의 행은 x_k, g(x_k)를 보여주고,
    # ea_k = |x_k - x_(k-1)| / x_k * 100 (같은 반복 번호끼리 비교)
    for k in range(0, max_iter + 1):

        gx = safe_g(x)

        if k == 0:
            ea = None
        else:
            ea = abs((x - x_prev) / x) * 100.0 if x != 0.0 else None
            if ea_first is None:
                ea_first = ea

        if x_true is not None:
            et = true_error(x)
            et_ratio = (
                et / et_prev
                if (k > 0 and et_prev not in (None, 0.0))
                else None
            )
            et_prev = et

        if verbose:
            ea_str = (
                f"{ea:8.2f}"
                if ea is not None
                else f"{'--':>8}"
            )
            row = (
                f"{k:4d} | "
                f"{x:10.4f} | "
                f"{gx:10.4f} | "
                f"{ea_str}"
            )
            if x_true is not None:
                et_str = f"{et:8.2f}"
                ratio_str = (
                    f"{et_ratio * 100.0:12.2f}"
                    if et_ratio is not None
                    else f"{'--':>12}"
                )
                row += f" | {et_str} | {ratio_str}"
            print(row)

        if ea is not None and ea <= tol_percent:
            return x, k, ea

        x_prev = x
        x = gx

    # max_iter 안에 수렴하지 못한 경우: 오차가 줄지 않고 오히려
    # 커졌다면(발산 추세) 명확한 경고를 남긴다.
    if verbose and ea is not None and ea_first is not None and ea > ea_first:
        print(
            "\n[경고] 근사 상대오차(ea)가 반복할수록 줄어들지 않고 커지고 "
            "있습니다 — 이 g(x)에서는 반복법이 발산하는 것으로 보입니다 "
            "(수렴 조건 |g'(x*)| < 1 을 만족하는지 확인하세요)."
        )

    return x_prev, max_iter, ea

### [Algorithm] Wegstein법

In [ ]:
import math

def wegstein_intersection(g, x0, tol_percent=0.01, max_iter=10_000,
                          fallback_when_s1=True, eps=1e-15, verbose=True,
                          x_true=None):
    """
    Wegstein (두 점을 지나는 직선과 y=x의 교점) 방식.
    - 첫 스텝: x1 = g(x0) 로 시작(두 점 확보).
    - k>=2: s_k로 직선 기울기 계산 후 교점 x_{k+1} 계산.

    Parameters
    ----------
    g : callable         고정점식 g(x)
    x0 : float           초기 추정값
    tol_percent : float  근사 백분율 상대오차(%) 허용치
    max_iter : int       최대 반복 횟수
    fallback_when_s1 : bool  s_k≈1(분모→0)일 때 x_{k+1}=g(x_k)로 폴백
    eps : float          0 나눗셈/불안정 방지용 작은 값
    verbose : bool       반복 표 출력
    x_true : float, optional
        참값. 주어지면 각 반복마다 참백분율상대오차(et)와, 이전 반복 대비
        현재 반복의 참오차 비율(et_ratio = et_k / et_(k-1), 백분율)도 함께
        출력한다. 주어지지 않으면(기본값 None) 이 두 값은 계산도 출력도
        하지 않는다.

    Returns
    -------
    root, iters, ea_percent
    """

    def true_error(xv):
        denom = x_true if abs(x_true) > eps else eps
        return abs((x_true - xv) / denom) * 100.0

    if verbose:
        header = (
            f"{'iter':>4} | {'x_prev':>10} | {'x_curr':>10} | "
            f"{'g(prev)':>10} | {'g(curr)':>10} | {'s_k':>10} | "
            f"{'x_new':>10} | {'ea(%)':>8}"
        )
        if x_true is not None:
            header += f" | {'et(%)':>8} | {'et_ratio(%)':>12}"
        print(header)
        print("-" * len(header))

    # 초기 점 확보: x0 하나만 주어지면 x1 = g(x0)로 두 번째 점을 만든다
    x_prev = float(x0)
    g_prev = g(x_prev)
    x_curr = g_prev  # x1 = g(x0)
    g_curr = g(x_curr)

    et_prev = None

    # ========================================================
    # Iteration 0: 초기값 x0와 그로부터 얻은 x1(=g(x0))을 미리보기로 표시
    # ========================================================

    x_new0 = x_curr  # 아직 s_k를 계산할 두 번째 쌍이 없으므로 x1 자체가 미리보기 값

    if x_true is not None:
        et_prev = true_error(x_new0)

    if verbose:
        row = (
            f"{0:4d} | {x_prev:10.4f} | {x_curr:10.4f} | "
            f"{g_prev:10.4f} | {g_curr:10.4f} | {'--':>10} | "
            f"{x_new0:10.4f} | {'--':>8}"
        )
        if x_true is not None:
            row += f" | {et_prev:8.2f} | {'--':>12}"
        print(row)

    # ========================================================
    # Iteration 1: iteration 0에서 얻은 x1을 그대로 사용
    # ========================================================

    denom = max(abs(x_curr), eps)
    ea = abs((x_curr - x_prev) / denom) * 100.0

    et = None
    if x_true is not None:
        et = true_error(x_curr)
        et_ratio = (
            et / et_prev if et_prev not in (None, 0.0) else None
        )
        et_prev = et

    if verbose:
        row = (
            f"{1:4d} | {x_prev:10.4f} | {x_curr:10.4f} | "
            f"{g_prev:10.4f} | {g_curr:10.4f} | {'--':>10} | "
            f"{x_curr:10.4f} | {ea:8.2f}"
        )
        if x_true is not None:
            ratio_str = (
                f"{et_ratio * 100.0:12.2f}" if et_ratio is not None else f"{'--':>12}"
            )
            row += f" | {et:8.2f} | {ratio_str}"
        print(row)

    if ea <= tol_percent:
        return x_curr, 1, ea

    # 이후 스텝들
    for k in range(2, max_iter + 1):
        # s_k = (gk - gk-1)/(xk - xk-1)
        denom_s = (x_curr - x_prev)
        if abs(denom_s) < eps:
            s = float('inf')
        else:
            s = (g_curr - g_prev) / denom_s

        # x_{k+1} = (gk - s_k*xk)/(1 - s_k)
        denom_q = (1.0 - s)
        if (abs(denom_q) < eps) or math.isinf(s) or math.isnan(s):
            # s≈1 (또는 불안정): 폴백
            if fallback_when_s1:
                x_new = g_curr
            else:
                raise ZeroDivisionError("Wegstein: s_k≈1로 교점 계산이 불안정합니다.")
        else:
            x_new = (g_curr - s * x_curr) / denom_q

        # 오차
        denom = max(abs(x_new), eps)
        ea = abs((x_new - x_curr) / denom) * 100.0

        if x_true is not None:
            et = true_error(x_new)
            et_ratio = (
                et / et_prev if et_prev not in (None, 0.0) else None
            )
            et_prev = et

        if verbose:
            s_str = f"{s:10.4f}" if math.isfinite(s) else f"{'inf':>10}"
            row = (
                f"{k:4d} | {x_prev:10.4f} | {x_curr:10.4f} | "
                f"{g_prev:10.4f} | {g_curr:10.4f} | {s_str} | "
                f"{x_new:10.4f} | {ea:8.2f}"
            )
            if x_true is not None:
                ratio_str = (
                    f"{et_ratio * 100.0:12.2f}" if et_ratio is not None else f"{'--':>12}"
                )
                row += f" | {et:8.2f} | {ratio_str}"
            print(row)

        if ea <= tol_percent:
            return x_new, k, ea

        # 다음 반복을 위한 갱신
        x_prev, g_prev = x_curr, g_curr
        x_curr = x_new
        g_curr = g(x_curr)

    return x_curr, max_iter, ea

### [Algorithm] Newton-Raphson 법

In [ ]:
import math

def newton_raphson(f, dfunc=None, x0=0.5, tol_percent=0.01, f_tol=0.0,
                   max_iter=10_000, h=1.e-6, verbose=True, x_true=None):
    """
    Newton-Raphson (generic)
    - f: 목적함수 f(x)
    - dfunc: f'(x) (없으면 중심차분으로 수치미분)
    - x0: 초기 추정값
    - tol_percent: 근사 백분율 상대오차(%) 허용치
    - f_tol: |f(x)| 허용치 (0이면 미사용)
    - max_iter: 최대 반복 횟수
    - h: 수치미분 간격
    - verbose: 반복 과정 출력 여부
    - x_true: 참값. 주어지면 각 반복마다 참백분율상대오차(et)와, 이전 반복
      대비 현재 반복의 참오차 비율(et_ratio = et_k / et_(k-1), 백분율)도
      함께 출력한다. 주어지지 않으면(기본값 None) 이 두 값은 계산도
      출력도 하지 않는다.

    반환: (root, iters, ea_percent)
    """
    def num_deriv(x):
        return (f(x + h) - f(x)) / h

    deriv = dfunc if dfunc is not None else num_deriv

    eps = 1e-15  # 0 나눗셈 방지

    def true_error(xv):
        denom = x_true if abs(x_true) > eps else eps
        return abs((x_true - xv) / denom) * 100.0

    if verbose:
        header = (
            f"{'iter':>4} | {'x_old':>10} | {'f(x_old)':>14} | "
            f"{'df(x_old)':>14} | {'x_new':>10} | {'ea(%)':>8}"
        )
        if x_true is not None:
            header += f" | {'et(%)':>8} | {'et_ratio(%)':>12}"
        print(header)
        print("-" * len(header))

    x_old = float(x0)
    et_prev = None

    # ========================================================
    # Iteration 0: 초기값 x0와 첫 갱신값 미리보기만 표시(오차는 계산하지 않음)
    # ========================================================

    fx = f(x_old)
    dfx = deriv(x_old)
    if abs(dfx) < eps:
        raise ZeroDivisionError("도함수가 0에 너무 가깝습니다. 다른 초기값을 사용하세요.")
    x_new = x_old - fx / dfx

    if x_true is not None:
        et_prev = true_error(x_new)

    if verbose:
        row = (
            f"{0:4d} | {x_old:10.4f} | {fx:14.10f} | "
            f"{dfx:14.10f} | {x_new:10.4f} | {'--':>8}"
        )
        if x_true is not None:
            row += f" | {et_prev:8.2f} | {'--':>12}"
        print(row)

    # ========================================================
    # 반복 (iteration 0에서 미리 계산한 fx, dfx, x_new을 그대로 사용)
    # ========================================================

    for k in range(1, max_iter + 1):

        denom = max(abs(x_new), eps)
        ea = abs((x_new - x_old) / denom) * 100.0

        et = None
        if x_true is not None:
            et = true_error(x_new)
            et_ratio = (
                et / et_prev if et_prev not in (None, 0.0) else None
            )
            et_prev = et

        if verbose:
            row = (
                f"{k:4d} | {x_old:10.4f} | {fx:14.10f} | "
                f"{dfx:14.10f} | {x_new:10.4f} | {ea:8.2f}"
            )
            if x_true is not None:
                ratio_str = (
                    f"{et_ratio * 100.0:12.2f}" if et_ratio is not None else f"{'--':>12}"
                )
                row += f" | {et:8.2f} | {ratio_str}"
            print(row)

        # 수렴 판정
        if (ea <= tol_percent) or (f_tol > 0.0 and abs(f(x_new)) <= f_tol):
            return x_new, k, ea

        # 다음 반복 준비
        x_old = x_new
        fx = f(x_old)
        dfx = deriv(x_old)
        if abs(dfx) < eps:
            raise ZeroDivisionError("도함수가 0에 너무 가깝습니다. 다른 초기값을 사용하세요.")
        x_new = x_old - fx / dfx

    return x_new, max_iter, ea

### [Algorithm] 할선법(Secant Method)

In [ ]:
import numpy as np

def secant_method(func, x0, x1, tol_percent=0.01, f_tol=0.0,
                  max_iter=10_000, eps=1e-15, verbose=True, x_true=None):
    """
    Secant method (generic)

    Parameters
    ----------
    func : callable
        목적함수 f(x)
    x0, x1 : float
        초기 두 추정값
    tol_percent : float
        근사 백분율 상대오차 허용치 (%)
    f_tol : float
        |f(x)| 허용치 (0이면 미사용)
    max_iter : int
        최대 반복 횟수
    eps : float
        분모가 0되는 것 방지용 작은 값
    verbose : bool
        반복 과정 출력 여부
    x_true : float, optional
        참값. 주어지면 각 반복마다 참백분율상대오차(et)와, 이전 반복 대비
        현재 반복의 참오차 비율(et_ratio = et_k / et_(k-1), 백분율)도 함께
        출력한다. 주어지지 않으면(기본값 None) 이 두 값은 계산도 출력도
        하지 않는다.

    Returns
    -------
    root : float
    iters : int
    ea : float
    """
    if not callable(func):
        raise TypeError(f"'func' must be callable; got {type(func).__name__}")

    def true_error(xv):
        denom = x_true if abs(x_true) > eps else eps
        return abs((x_true - xv) / denom) * 100.0

    if verbose:
        header = (
            f"{'iter':>4} | {'x_prev':>10} | {'x_curr':>10} | "
            f"{'f(prev)':>14} | {'f(curr)':>14} | {'x_new':>10} | "
            f"{'ea(%)':>8}"
        )
        if x_true is not None:
            header += f" | {'et(%)':>8} | {'et_ratio(%)':>12}"
        print(header)
        print("-" * len(header))

    f0 = func(x0)
    f1 = func(x1)
    et_prev = None

    # ========================================================
    # Iteration 0: 초기 두 점 x0, x1과 첫 갱신값 미리보기만 표시
    # ========================================================

    denom = (f1 - f0)
    if abs(denom) < eps:
        raise ZeroDivisionError("Secant method denominator too small; f(x1)-f(x0)≈0.")
    x_new = x1 - f1 * (x1 - x0) / denom

    if x_true is not None:
        et_prev = true_error(x_new)

    if verbose:
        row = (
            f"{0:4d} | {x0:10.4f} | {x1:10.4f} | "
            f"{f0:14.10f} | {f1:14.10f} | {x_new:10.4f} | {'--':>8}"
        )
        if x_true is not None:
            row += f" | {et_prev:8.2f} | {'--':>12}"
        print(row)

    # ========================================================
    # 반복 (iteration 0에서 미리 계산한 x_new을 그대로 사용)
    # ========================================================

    for k in range(1, max_iter + 1):

        f_new = func(x_new)

        denom_ea = max(abs(x_new), eps)
        ea = abs((x_new - x1) / denom_ea) * 100.0

        et = None
        if x_true is not None:
            et = true_error(x_new)
            et_ratio = (
                et / et_prev if et_prev not in (None, 0.0) else None
            )
            et_prev = et

        if verbose:
            row = (
                f"{k:4d} | {x0:10.4f} | {x1:10.4f} | "
                f"{f0:14.10f} | {f1:14.10f} | {x_new:10.4f} | {ea:8.2f}"
            )
            if x_true is not None:
                ratio_str = (
                    f"{et_ratio * 100.0:12.2f}" if et_ratio is not None else f"{'--':>12}"
                )
                row += f" | {et:8.2f} | {ratio_str}"
            print(row)

        # 수렴 판정
        if (ea <= tol_percent) or (f_tol > 0.0 and abs(f_new) <= f_tol):
            return x_new, k, ea

        # 다음 반복 준비
        x0, f0 = x1, f1
        x1, f1 = x_new, f_new
        denom = (f1 - f0)
        if abs(denom) < eps:
            raise ZeroDivisionError("Secant method denominator too small; f(x1)-f(x0)≈0.")
        x_new = x1 - f1 * (x1 - x0) / denom

    return x_new, max_iter, ea

- 예제 6.1 - 단순 고정점 반복법

$$
f(x) = e^{-x}-x
$$

In [ ]:
x_true = 0.567143290409783872999968662210355549753815787186512508135131

In [ ]:
# g(x) 정의: x = exp(-x)
g = lambda x: math.exp(-x)

root, iters, ea = fixed_point(g, x0=0.0, tol_percent=0.01, max_iter=10, verbose=True, x_true=x_true)

print("\nResult:")
print(f"x* ≈ {root:.10f}, iterations = {iters}, final ea ≈ {ea:.6f}%")

In [ ]:
# g(x) 정의: x = - ln(x)
g = lambda x: - math.log(x)

root, iters, ea = fixed_point(g, x0=0.5, tol_percent=0.01, max_iter=3, verbose=True, x_true=x_true)

print("\nResult:")
print(f"x* ≈ {root:.10f}, iterations = {iters}, final ea ≈ {ea:.6f}%")